In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark import SparkContext, SparkConf
import re

def normalize_words(text):
    return re.compile(r'\W+',re.UNICODE).split(text.lower())

In [2]:
spark_conf = (SparkConf()
    .setMaster('local[1]')
    .setAppName('18')
    .set('spark.driver.host', '127.0.0.1')
    .set('spark.driver.bindAddress', '127.0.0.1')
    .set('spark.pyspark.python', sys.executable)
    .set('spark.python.worker.reuse', 'false')
    .set('spark.ui.enabled', 'false'))

In [3]:
spark = SparkContext.getOrCreate(conf=spark_conf)

In [4]:
data_path = r'C:\datas\3rd_week_data\fakefriends.csv'
lines = spark.textFile(data_path)

In [5]:
lines.take(5)

['0,Will,33,385',
 '1,Jean-Luc,26,2',
 '2,Hugh,55,221',
 '3,Deanna,40,465',
 '4,Quark,68,21']

- `lines`: **index**, **이름**, **나이**, **친구의 수**
- **목표**: 각 나이별 친구의 수 평균 구하기

## 2. **Transformation**
- `map`을 사용하여, 필요한 데이터를 (key, value)로 추출
- 나이별 친구의 수를 구하고자 하므로, 이름과 id를 제외한 **나이**와 **친구의 수**만 추출함
- 이 때, value는 집계가 되어야 하는 값이므로, value는 **친구의 수**, key는 집계된 각 값을 대표하는 값이므로 **나이**로 설정함

In [6]:
def parseLine(line):
    # 콤마(,)를 기준으로 분리
    fields = line.split(',')
    # 세 번째 요소인 "나이"와 네 번째 요소인 "친구의 수"만 반환
    # 이 때, "친구의 수"는 집계(연산)가 필요하므로, numeric 값(int)으로 변환함
    return fields[2], int(fields[3])
    # key: fields[2], value: int(fields[3])

- `map`에서 사용되는 parsing 함수가 어떠한 형태로 값을 반환하느냐에 따라 **Key**와 **Value**를 구분함
- 두 개로 분리되는 경우, 앞의 값이 Key, 뒤의 값이 Value가 됨
> - ex> `return a,b` -> key: `a`, value: `b`
> - ex> `return (a,b)` -> key: `a`, value: `b`
> - ex> `return a,b,c` -> key: `없음`, value: `a,b,c`
> - ex> `return a,(b,c)` -> key: `a`, value: `(b,c)`
> - ex> `return (a,b),c` -> key: `(a,b)`, value: `c`

In [7]:
rdd = lines.map(parseLine)

In [8]:
rdd.take(5)

[('33', 385), ('26', 2), ('55', 221), ('40', 465), ('68', 21)]

## 3. **Action** (`reduceByKey`)
- RDD의 `key`를 기준으로 집계
- `reduceByKey`의 인자는 **key**를 기준으로 **value를 집계**할 수 있는 함수임
- `reduceByKey` 인자인 **집계함수의 인자**는 **두 개의 RDD value**가 됨
- 동일한 **Key**를 만났을 때, 그 **2개의 value**를 어떻게 처리(집계)할 것인가를 함수를 통해 정의함

In [9]:
sum_age = rdd.reduceByKey(lambda val1, val2: val1+val2)
# 동일한 key를 갖는 두 RDD의 value인 val1과 val2를 더함

In [10]:
sum_age.take(5)

[('33', 3904), ('26', 4115), ('55', 3842), ('40', 4264), ('68', 2696)]

## 4. **각 나이별 친구의 수 평균 찾기**

### 4-1. `mapValues` 이용
- **전략**: 각 나이별 친구의 수를 더함과 동시에 나이(key)에 해당하는 라인의 수를 카운트함 (즉, 친구수 합계와 동시에 그 나이의 사람 수 카운트)
- `mapValues()`: key를 제외한 **value만 변경**하는 transformation, 인자는 **value를 변경하는 함수**
- `mapValues`에 적용된 **함수 인자는 RDD의 key를 제외한 value**임에 주의할 것

In [11]:
# (1) val: 친구의 수 -> val: (친구의 수, 1) 로 변경
tmp_rdd = rdd.mapValues(lambda val: (val,1))
tmp_rdd.take(5)

[('33', (385, 1)),
 ('26', (2, 1)),
 ('55', (221, 1)),
 ('40', (465, 1)),
 ('68', (21, 1))]

In [12]:
# (2) reduceByKey를 통해 value인 친구의 수와 '1'을 각각 더함
# 집계된 결과는 (나이별 친구의 수 합, 해당 나이의 사람 수)
# ex> (33살인 사람의 친구 수의 합, 33살인 사람의 수) -> 첫번째 값에서 두번째 값을 나누면 평균을 구할 수 있음
sum_age_count = tmp_rdd.reduceByKey(lambda val1, val2: (val1[0]+val2[0], val1[1]+val2[1]))
# 각 value는 tuple로 (친구의 수, 1)이므로 value[0]과 value[1]을 각각 더함
sum_age_count.take(5)

[('33', (3904, 12)),
 ('26', (4115, 17)),
 ('55', (3842, 13)),
 ('40', (4264, 17)),
 ('68', (2696, 10))]

In [13]:
# (3) value의 첫번째 값(친구 수의 합)을 두번째 값(해당 나이 사람의 수)으로 나누어 평균값을 구함
avg_friends = sum_age_count.mapValues(lambda val: val[0]/val[1])
avg_friends.take(5)

[('33', 325.3333333333333),
 ('26', 242.05882352941177),
 ('55', 295.53846153846155),
 ('40', 250.8235294117647),
 ('68', 269.6)]

### 4-2. `countByKey` 이용
- `RDD.countByKey()`: 각 **Key**를 포함한 data의 수를 dict type으로 반환

In [14]:
count_age = rdd.countByKey()
print(type(count_age))
# count_age의 결과와 sum_age(각 나이별 친구 수의 합)을 결합하는 전략
# 그러나 sum_age는 RDD, count_age는 dict 타입으로 결합이 불가능함
# count_age를 RDD 타입으로의 변환이 필요

<class 'collections.defaultdict'>


- `SparkContext.parallelize()`: 입력된 list 및 iterable 객체를 RDD로 변환 `SparkContext.parallelize()`: 입력된 list 및 iterable 객체를 RDD로 변환
> - `parallelize(dict)`는 dict의 key 값만 RDD로 생성
> - `parallelize(dict.items())`는 key와 value를 각각 RDD의 key와 value로 생성

In [15]:
count_age = spark.parallelize(rdd.countByKey().items())
count_age.take(5)

[('33', 12), ('26', 17), ('55', 13), ('40', 17), ('68', 10)]

- `RDD1.join(RDD2)`: 2개의 RDD를 key를 기준으로 결합

In [16]:
sum_age_count = sum_age.join(count_age)
sum_age_count.take(5)
# key(나이), (친구수의 합, 해당 나이인 사람의 수)

[('33', (3904, 12)),
 ('26', (4115, 17)),
 ('55', (3842, 13)),
 ('40', (4264, 17)),
 ('68', (2696, 10))]

In [21]:
# mapValues를 통해 평균값을 구함
avg_friends = sum_age_count.mapValues(lambda val: val[0]/val[1])
avg_friends.take(5)

[('33', 325.3333333333333),
 ('26', 242.05882352941177),
 ('55', 295.53846153846155),
 ('40', 250.8235294117647),
 ('68', 269.6)]

In [22]:
spark.stop()